# Portfolio model summary

This notebook explains the final merchant portfolio model and provides compact checks of the delivered dashboard outputs. It uses the final CSV files only and does not rebuild or change the model.

The model selects exactly **100 merchants**. Results are available for one overall portfolio and five category portfolios at five minimum expected-revenue retention levels: 100%, 99.5%, 99%, 98%, and 95%.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

# Works when run from either the repository root or the code directory.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
FINAL_DIR = PROJECT_ROOT / "handoff" / "final"

DASHBOARD_FILES = {
    "Overall": "portfolio_dashboard.csv",
    "Arts, Crafts, Gifts & Entertainment": "arts_portfolio_dashboard.csv",
    "Automotive, Outdoor & Recreational Equipment": "auto_portfolio_dashboard.csv",
    "Fashion, Jewellery & Personal Care": "fashion_portfolio_dashboard.csv",
    "Home, Garden & General Supplies": "home_portfolio_dashboard.csv",
    "Technology, Digital & Communications": "tech_portfolio_dashboard.csv",
}

dashboards = {
    label: pd.read_csv(FINAL_DIR / filename, dtype={"merchant_abn": "string"})
    for label, filename in DASHBOARD_FILES.items()
}

pd.DataFrame(
    {
        "dashboard": label,
        "rows": len(frame),
        "portfolios": frame["portfolio_id"].nunique(),
        "merchants_per_portfolio": int(frame.groupby("portfolio_id").size().min()),
    }
    for label, frame in dashboards.items()
)

,dashboard,rows,portfolios,merchants_per_portfolio
0,Overall,500,5,100
1,"Arts, Crafts, Gifts & Entertainment",500,5,100
2,"Automotive, Outdoor & Recreational Equipment",500,5,100
3,"Fashion, Jewellery & Personal Care",500,5,100
4,"Home, Garden & General Supplies",500,5,100
5,"Technology, Digital & Communications",500,5,100


## 1. Expected revenue

The dashboard calls the model's revenue measure **expected monthly gross fees**. For merchant $i$:

$$E_i = \left(\frac{1}{M}\sum_{m=1}^{M} \text{monthly sales}_{i,m}\right) \times \frac{\text{take rate}_i}{100}$$

The delivered snapshot uses 19 complete calendar months from March 2021 through September 2022. Genuine zero-sales months remain in the average. A merchant is eligible only when it has positive sales in each of the two most recent three-month blocks.

For a selected portfolio represented by binary indicators $z_i$, expected monthly portfolio revenue is:

$$E_P = \sum_i z_i E_i$$

The revenue-only baseline is the 100 eligible merchants with the largest $E_i$. Each optimized portfolio must retain at least a chosen fraction $r$ of that baseline revenue:

$$E_P \ge r E_{baseline}$$

The `retention` field is this minimum constraint, while `portfolio_expected_fee_retention` is the amount actually achieved. Within each selected portfolio, merchants are sorted from highest to lowest `merchant_expected_monthly_gross_fees`; ABN is the deterministic tie-breaker.

## 2. Stability (cash-flow variability)

Stability is measured from **26 complete Monday–Sunday weeks** of merchant gross-fee history. Let $x_t$ be the vector of merchant fees in week $t$. The model estimates a Ledoit–Wolf shrinkage covariance matrix:

$$\widehat{\Sigma}_{shrunk} = (1-\alpha)\widehat{\Sigma}_{sample} + \alpha\,\mathrm{diag}(\widehat{\Sigma}_{sample})$$

Shrinkage reduces unstable off-diagonal covariance estimates while retaining each merchant's variance. For selection vector $z$:

$$SD_P = \sqrt{z^T\widehat{\Sigma}_{shrunk}z}$$

$$CV_P = \frac{SD_P}{\sum_i z_i\bar{x}_i}$$

Here $\bar{x}_i$ is merchant $i$'s historical weekly mean. The coefficient of variation (CV) makes volatility comparable relative to portfolio size: a **lower CV means more stable weekly fee generation**. CV is a ratio and is not constrained to the interval $[0,1]$.

## 3. Customer concentration

Customer concentration measures whether expected portfolio fees depend heavily on a small number of customers. Historical customer–merchant gross fees are rescaled merchant by merchant so that each merchant's customer allocations sum to its expected monthly fees. This gives exposure $A_{ci}$ from customer $c$ to merchant $i$. Shared customers are combined across all selected merchants:

$$F_c(z) = \sum_i A_{ci}z_i$$

Customer shares and the Herfindahl–Hirschman Index (HHI) are then:

$$s_c(z) = \frac{F_c(z)}{\sum_c F_c(z)}, \qquad HHI_P = \sum_c s_c(z)^2$$

HHI lies between 0 and 1. **Lower HHI means customer exposure is more diversified**; higher HHI means more reliance on a smaller customer base. The reciprocal, $1/HHI$, can be interpreted as the effective number of equally weighted customers.

## 4. Combined risk score and portfolio selection

CV and HHI use different scales, so each is divided by its value for the revenue-only top-100 baseline. With variability weight $w$ and concentration weight $1-w$:

$$Score_P = 100\left[w\frac{CV_P}{CV_{baseline}} + (1-w)\frac{HHI_P}{HHI_{baseline}}\right]$$

The delivered model uses $w=0.5$, so stability and customer concentration receive equal weight. The revenue-only baseline has a score of 100. A score below 100 indicates lower combined in-sample risk than that baseline. The score is **not a probability**.

For each retention level, the optimizer chooses exactly 100 merchants, respects the expected-revenue floor, and minimizes the combined score. `portfolio_score_lower_bound`, `portfolio_optimality_gap`, and `portfolio_certified` describe optimization evidence; certification concerns the numerical solution tolerance, not future performance.

## 5. Revenue and customer-concentration correlation matrix

The dashboard repeats portfolio metrics on each merchant row. The next cell first keeps one row per portfolio and then calculates the **Pearson correlation across the 30 delivered portfolios** (six dashboard scopes × five retention levels). This avoids incorrectly weighting a portfolio 100 times. Correlation is descriptive only and does not imply causation.

In [2]:
portfolio_rows = pd.concat(
    [
        frame.drop_duplicates("portfolio_id").assign(dashboard=label)
        for label, frame in dashboards.items()
    ],
    ignore_index=True,
)

correlation_data = portfolio_rows[[
    "portfolio_expected_monthly_gross_fees",
    "portfolio_customer_hhi",
]].rename(
    columns={
        "portfolio_expected_monthly_gross_fees": "Expected monthly revenue",
        "portfolio_customer_hhi": "Customer concentration (HHI)",
    }
)

correlation_matrix = correlation_data.corr(method="pearson")
display(correlation_matrix.round(3))

,Expected monthly revenue,Customer concentration (HHI)
Expected monthly revenue,1.000000,-0.588000
Customer concentration (HHI),-0.588000,1.000000


## 6. Top-100 rankings: first five merchants

Each dashboard contains five different top-100 portfolios—one per retention setting. The display below defaults to the 100% retention portfolio. Change `SELECTED_RETENTION` to `0.995`, `0.99`, `0.98`, or `0.95` to inspect another portfolio. `head(5)` shows ranks 1–5 after an explicit rank sort.

In [3]:
SELECTED_RETENTION = 1.0

RANKING_COLUMNS = [
    "rank_by_expected_fees",
    "merchant_abn",
    "merchant_name",
    "merchant_category",
    "merchant_expected_monthly_gross_fees",
]

def ranking_head(frame, retention=SELECTED_RETENTION):
    selected = frame.loc[np.isclose(frame["retention"], retention)].copy()
    if len(selected) != 100:
        raise ValueError(f"Expected 100 merchants at retention={retention}, found {len(selected)}")
    return selected.sort_values("rank_by_expected_fees")[RANKING_COLUMNS].head(5)

print(f"Overall portfolio — retention {SELECTED_RETENTION:.1%}")
display(ranking_head(dashboards["Overall"]))

Overall portfolio — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_gross_fees
0,1,79827781481,Amet Risus Inc.,"furniture, home furnishings and equipment shop...","33,161.999483"
1,2,48534649627,Dignissim Maecenas Foundation,"opticians, optical goods, and eyeglasses","31,088.087839"
2,3,32361057556,Orci In Consequat Corporation,"gift, card, novelty, and souvenir shops","31,053.347465"
3,4,86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops","30,586.118562"
4,5,38700038932,Etiam Bibendum Industries,tent and awning shops,"30,106.623490"


In [4]:
for label, frame in dashboards.items():
    if label == "Overall":
        continue
    print(f"{label} — retention {SELECTED_RETENTION:.1%}")
    display(ranking_head(frame))

Arts, Crafts, Gifts & Entertainment — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_gross_fees
0,1,32361057556,Orci In Consequat Corporation,"gift, card, novelty, and souvenir shops","31,053.347465"
1,2,45629217853,Lacus Consulting,"gift, card, novelty, and souvenir shops","29,233.139008"
2,3,63123845164,Odio Phasellus Institute,artist supply and craft shops,"28,213.797077"
3,4,64403598239,Lobortis Ultrices Company,"music shops - musical instruments, pianos, and...","27,879.515277"
4,5,94493496784,Dictum Phasellus In Institute,"gift, card, novelty, and souvenir shops","25,653.901521"


Automotive, Outdoor & Recreational Equipment — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_gross_fees
0,1,38700038932,Etiam Bibendum Industries,tent and awning shops,"30,106.623490"
1,2,96680767841,Ornare Limited,motor vehicle supplies and new parts,"28,873.185259"
2,3,89726005175,Est Nunc Consulting,tent and awning shops,"26,644.910948"
3,4,80551528183,Ac Ipsum LLC,tent and awning shops,"21,254.331719"
4,5,49891706470,Non Vestibulum Industries,tent and awning shops,"20,707.173292"


Fashion, Jewellery & Personal Care — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_gross_fees
0,1,48534649627,Dignissim Maecenas Foundation,"opticians, optical goods, and eyeglasses","31,088.087839"
1,2,86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops","30,586.118562"
2,3,49322182190,Gravida Mauris Incorporated,"watch, clock, and jewelry repair shops","24,839.190191"
3,4,19492220327,Commodo Ipsum Industries,"jewelry, watch, clock, and silverware shops","20,153.580080"
4,5,93558142492,Dolor Quisque Inc.,shoe shops,"15,058.810146"


Home, Garden & General Supplies — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_gross_fees
0,1,79827781481,Amet Risus Inc.,"furniture, home furnishings and equipment shop...","33,161.999483"
1,2,76767266140,Phasellus At Limited,"furniture, home furnishings and equipment shop...","21,643.250122"
2,3,43186523025,Lorem Ipsum Sodales Industries,"florists supplies, nursery stock, and flowers","20,142.046513"
3,4,49212265466,Auctor Company,"florists supplies, nursery stock, and flowers","18,225.235162"
4,5,57757792876,Pretium Et LLC,"stationery, office supplies and printing and w...","17,480.611380"


Technology, Digital & Communications — retention 100.0%


,rank_by_expected_fees,merchant_abn,merchant_name,merchant_category,merchant_expected_monthly_gross_fees
0,1,21439773999,Mauris Non Institute,"cable, satellite, and other pay television and...","28,634.957886"
1,2,82368304209,Nec Incorporated,telecom,"26,846.695224"
2,3,35909341340,Arcu Sed Eu Incorporated,"computer programming, data processing, and int...","22,760.744646"
3,4,72472909171,Nullam Consulting,"digital goods: books, movies, music","22,564.011261"
4,5,45433476494,Adipiscing Elit Foundation,"computers, computer peripheral equipment, and ...","20,784.249927"


## Interpretation notes

- Revenue is an expectation based on historical averages, not a guaranteed forecast.
- CV and HHI describe different risks: weekly cash-flow instability and dependence on shared customers.
- Relaxing the revenue-retention floor gives the optimizer more freedom to reduce risk, but actual achieved revenue retention may be above the minimum.
- Portfolio-level fields are repeated on all 100 merchant rows. Use one row per `portfolio_id` for portfolio analysis; only `merchant_expected_monthly_gross_fees` should be summed within a portfolio.
- The rankings order merchants inside an already selected portfolio by expected fees. They are not individual merchant risk rankings.